<a href="https://colab.research.google.com/github/waqarch-tech/YoungDevInterns_Artificial-Intelligence_Tasks/blob/main/Deploy_machine_learning_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
import kagglehub
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# 1. LOAD DATA
print("Loading data...")
path = kagglehub.dataset_download("mukundsavaliya/california-housing-data")
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

# 2. PREPROCESS (Creating X_scaled and y_class)
print("Preprocessing...")
df_encoded = pd.get_dummies(df)
imputer = SimpleImputer(strategy='median')
df_imputed = pd.DataFrame(imputer.fit_transform(df_encoded), columns=df_encoded.columns)

# Define X (features)
X = df_imputed.drop('median_house_value', axis=1)

# Define y_class (target) - 1 if above median price, else 0
y_class = (df_imputed['median_house_value'] > df_imputed['median_house_value'].median()).astype(int)

# Scale the data (This creates X_scaled)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. SPLIT DATA (This creates X_train, y_train, etc.)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_class, test_size=0.2, random_state=42)

print("SUCCESS: X_train and y_train are now defined and ready!")

Loading data...
Using Colab cache for faster access to the 'california-housing-data' dataset.
Preprocessing...
SUCCESS: X_train and y_train are now defined and ready!


In [38]:
import joblib
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel
from sklearn.ensemble import RandomForestClassifier

# --- 1. PREPARE THE MODEL FOR DEPLOYMENT ---
# We briefly retrain to ensure 'model' and 'scaler' exist in this session
print("Preparing model and scaler for deployment...")
model = RandomForestClassifier(n_estimators=10, random_state=42)
model.fit(X_train, y_train)

# Save the assets to Colab's local storage
joblib.dump(model, 'housing_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

# --- 2. DEFINE THE WEB SERVICE ---
app = FastAPI(title="California Housing API")

class PredictionInput(BaseModel):
    features: list # Expects a list of floats matching the dataset columns

@app.get("/")
def health_check():
    return {"status": "Online", "model": "RandomForestClassifier"}

@app.post("/predict")
def predict(input_data: PredictionInput):
    # Load assets from disk (simulating a real server environment)
    clf = joblib.load('housing_model.pkl')
    scl = joblib.load('scaler.pkl')

    # Process and predict
    raw_data = np.array(input_data.features).reshape(1, -1)
    scaled_data = scl.transform(raw_data)
    prediction = clf.predict(scaled_data)

    return {
        "prediction": int(prediction[0]),
        "label": "Expensive" if prediction[0] == 1 else "Affordable"
    }

print("\nTask 3 Code is ready. Model saved as 'housing_model.pkl'.")

Preparing model and scaler for deployment...

Task 3 Code is ready. Model saved as 'housing_model.pkl'.


In [39]:
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

# This is the actual Web Service
app = FastAPI()

# Create a class to define what the input data should look like
class HouseFeatures(BaseModel):
    features: list  # e.g., [longitude, latitude, housing_median_age, total_rooms, ...]

@app.post("/predict")
def predict_housing_class(input_data: HouseFeatures):
    # 1. Load the saved assets
    loaded_model = joblib.load('housing_model.pkl')
    loaded_scaler = joblib.load('scaler.pkl')

    # 2. Process input
    data = np.array(input_data.features).reshape(1, -1)
    scaled_data = loaded_scaler.transform(data)

    # 3. Predict
    prediction = loaded_model.predict(scaled_data)

    return {"is_expensive": int(prediction[0])}

In [40]:
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

# Load your Colab-generated assets
model = joblib.load('housing_model.pkl')
scaler = joblib.load('scaler.pkl')

app = FastAPI(title="California Housing API")

class HouseInput(BaseModel):
    # Expects a list of the 8-10 features used during training
    features: list

@app.post("/predict")
def predict_price_category(data: HouseInput):
    # 1. Convert input to the format the model expects
    input_array = np.array(data.features).reshape(1, -1)

    # 2. Scale the data (CRITICAL: model will fail if you skip this)
    scaled_data = scaler.transform(input_array)

    # 3. Get result
    prediction = model.predict(scaled_data)[0]

    return {
        "is_expensive": int(prediction),
        "result": "High Value" if prediction == 1 else "Standard Value"
    }

In [41]:
# This cell creates the Dockerfile needed for cloud deployment
%%writefile Dockerfile
FROM python:3.10-slim

# Set the working directory inside the container
WORKDIR /app

# Copy the model and scaler you see in your file sidebar (image_0bd2f6.png)
COPY housing_model.pkl .
COPY scaler.pkl .
COPY main.py .

# Install dependencies for the web service
RUN pip install fastapi uvicorn joblib scikit-learn numpy

# Expose the port for the web service
EXPOSE 8080

# Command to run the FastAPI server
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"]

Overwriting Dockerfile


In [42]:
# This cell creates the main.py file for the API
%%writefile main.py
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

# Load the assets you created in Tasks 1 & 2
model = joblib.load('housing_model.pkl')
scaler = joblib.load('scaler.pkl')

app = FastAPI()

class HouseInput(BaseModel):
    features: list

@app.post("/predict")
def predict(data: HouseInput):
    # Scale and predict
    input_data = np.array(data.features).reshape(1, -1)
    scaled_data = scaler.transform(input_data)
    prediction = model.predict(scaled_data)[0]

    return {
        "prediction": int(prediction),
        "status": "Expensive" if prediction == 1 else "Affordable"
    }

Writing main.py
